# Local sensitivity checks for PyMC posterior predictions

### 1. Setup and goal

Let
  - $X=(x_1,...,x_n),Y=(y_1,...,y_n)$ be observed data

  - $\theta \sim p(\theta|X,y)$ the posterior

  - $g(X,\theta)$ a posterior predictive or derived quantity


Here we look at how much posterior outputs change when the observed data are slightly perturbed, focusing on local effects (e.g. removing a single observation).
This is meant as a quick stability check, not a formal diagnostic.

### 2. Baseline Model

In [ ]:
import numpy as np
import pymc as pm
import pytensor.tensor as pt
import arviz as az

In [ ]:
rng = np.random.default_rng(42)

n = 50
X = np.linspace(0, 1, n)
y = 1.0 + 2.5 * X + rng.normal(0, 0.3, size=n)

In [ ]:
with pm.Model() as model:
    alpha = pm.Normal("alpha", 0, 5)
    beta = pm.Normal("beta", 0, 5)
    sigma = pm.HalfNormal("sigma", 1)

    mu = alpha + beta * X
    y_obs = pm.Normal("y_obs", mu, sigma, observed=y)

    idata = pm.sample(1000, tune=1000, chains=2, target_accept=0.9)

This defines a posterior $p(\theta|X,y)$ with
$\theta = (\alpha,\beta,\sigma)$

### 3. Posterior predictive functional

We consider the posterior predictive mean

$$
g(X)=\mathbb{E}_{\theta\sim p(\theta\mid X,y)}\!\left[y^{\mathrm{rep}}\mid X,\theta\right]
$$

approximated by Monte Carlo.

In [ ]:
with model:
    ppc = pm.sample_posterior_predictive(idata, var_names=["y_obs"])

In [ ]:
g_full = ppc.posterior_predictive["y_obs"].mean(
    dim=("chain", "draw")
).values

### 4. Local perturbation: leave one observation out

As a simple local perturbation, we remove a single data point
$$(X_{-i},y_{-i})=(X,y)\setminus (x_i​,y_i​).$$

This refit is used only as a reference for local sensitivity.

In [ ]:
X_m = X[:-1]
y_m = y[:-1]

In [ ]:
with pm.Model() as model_m:
    alpha = pm.Normal("alpha", 0, 5)
    beta = pm.Normal("beta", 0, 5)
    sigma = pm.HalfNormal("sigma", 1)

    mu = alpha + beta * X_m
    y_obs = pm.Normal("y_obs", mu, sigma, observed=y_m)

    idata_m = pm.sample(1000, tune=1000, chains=2, target_accept=0.9)

In [ ]:
with model_m:
    ppc_m = pm.sample_posterior_predictive(idata_m, var_names=["y_obs"])

In [ ]:
g_minus = ppc_m.posterior_predictive["y_obs"].mean(
    dim=("chain", "draw")
).values

### 5. A simple local sensitivity measure

We compare posterior outputs before and after the perturbation using

$$
S_i=\frac{\left\lVert g(X_{-i})-g(X)\mid X_{-i}\right\rVert_2}{\left\lVert g(X)\mid X_{-i}\right\rVert_2+\varepsilon}
$$

This measures relative change in posterior predictions; it is not an uncertainty estimate.

In [ ]:
def sensitivity(a, b, eps=1e-8):
    return np.linalg.norm(a - b) / (np.linalg.norm(a) + eps)

In [ ]:
S = sensitivity(g_full[:-1], g_minus)
S

### 6. Gradient-based local sensitivity (no refitting)

Instead of refitting, we can look at local linear sensitivity of the log-likelihood.

Let

$$l(y;\theta)=log p(y|\theta)$$

We examine gradients

$$\nabla l(y;\theta),$$

evaluated at the observed data.

In [ ]:
with model:
    logp = pm.logp(y_obs, y)

In [ ]:
grad_y = pt.grad(logp.sum(), y_obs)
grad_fn = pm.pytensorf.compile_pymc([], grad_y)

In [ ]:
grad_vals = grad_fn()
local_grad = np.abs(grad_vals)
local_grad

7. Relation to existing tools

- PSIS-LOO and $
\hat{k}
$ identify influential observations via importance sampling

- Here we directly examine changes in posterior-derived quantities

- Gradient-based sensitivity provides a cheap, autodiff-based local alternative

This notebook is complementary to existing ArviZ diagnostics.
(No API guarantees or statistical claims are implied.)
